# Practical 9 — Sentiment Analysis (VADER & TextBlob)

**Course:** NLP
**Name:**  <!-- fill in -->
**Date:**  <!-- fill in -->

## Aim
To score review sentiment using VADER and TextBlob, and to directly test — using the actual functions built in Practicals 1 and 2 — whether preprocessing (cleaning, stopword removal) helps or hurts sentiment scoring accuracy.

## Theory

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) is a lexicon + rule-based sentiment tool built specifically for informal, social-media-style text. It produces a `compound` score from -1 (most negative) to +1 (most positive), plus neg/neu/pos proportions. Critically, VADER's rules are explicitly built around signals like:
- **Punctuation emphasis** — "great!!!" scores more intensely positive than "great".
- **Capitalization emphasis** — "GREAT" scores more intensely than "great".
- **Negation** — "not good" is explicitly detected and flips/dampens the valence of "good", rather than the two words being scored independently.
- **Degree modifiers** — "very good" boosts beyond just "good".

**TextBlob** takes a simpler pattern-based approach: a lexicon of pre-scored adjectives combined with simpler rules, producing `polarity` (-1 to +1) and `subjectivity` (0 to 1).

Here's the direct test this practical runs: VADER's rules depend on exactly the signals — punctuation, capitalization, and intact negation words — that Practical 1's `clean_text()` strips out, and that Practical 2 showed can be destroyed by stopword removal. If VADER is genuinely built to use these signals, feeding it cleaned/stopword-removed text instead of raw text should measurably change its scores. This closes the loop opened all the way back in Practical 2, where the negation-loss risk was only a hypothesis about a downstream task — here it gets tested on an actual sentiment tool.

## Algorithm

1. **Before running anything below, write a one-sentence prediction**: do you expect VADER's scores to change meaningfully between raw and cleaned/stopword-removed text? Base it on what Practicals 1, 2, and 3 already showed about what gets lost in cleaning.
2. Score two punctuation/caps-heavy raw reviews with VADER, then score cleaned versions of the same reviews, and compare.
3. Take review 7 ("...would not recommend...") specifically, and score VADER on: the raw text, the cleaned text, and the stopword-removed text (using Practical 2's actual `remove_stopwords` function) — three versions, one review.
4. Score all 15 raw reviews with both VADER and TextBlob, bucket by sentiment, and compare the two tools' agreement.

In [1]:
import sys, os
sys.path.append(os.path.abspath("../python"))

import pandas as pd

import preprocessing
import tokenizer
import sentiment

pd.set_option("display.max_colwidth", None)

df = pd.read_csv("../datasets/sample_reviews.csv")
nltk_stops = preprocessing.get_nltk_stopwords()
print(f"Loaded {len(df)} reviews")


Loaded 15 reviews


### Step 1 — Your prediction

*Write 1-2 sentences here before running anything below: do you expect VADER's compound score to change meaningfully once punctuation/capitalization/negation words are stripped out? Why or why not, based on Practicals 1-3?*


### Step 2 — Punctuation/capitalization: raw vs cleaned

In [2]:
for review_id in [1, 8]:
    raw = df[df["id"] == review_id]["review"].iloc[0]
    cleaned = preprocessing.clean_text(raw)

    raw_score = sentiment.vader_score(raw)
    cleaned_score = sentiment.vader_score(cleaned)

    print(f"Review {review_id}")
    print(f"  RAW:     {raw}")
    print(f"  CLEANED: {cleaned}")
    print(f"  VADER compound (RAW):     {raw_score['compound']}")
    print(f"  VADER compound (CLEANED): {cleaned_score['compound']}")
    print("-" * 60)


Review 1
  RAW:     This movie was ABSOLUTELY fantastic!!! I've never seen anything like it   before.
  CLEANED: this movie was absolutely fantastic ive never seen anything like it before
  VADER compound (RAW):     0.6588
  VADER compound (CLEANED): 0.4182
------------------------------------------------------------
Review 8
  RAW:     WOW!!! Best film I've seen in years, hands down. 10/10 no notes.
  CLEANED: wow best film ive seen in years hands down no notes
  VADER compound (RAW):     0.8559
  VADER compound (CLEANED): 0.7783
------------------------------------------------------------


### Step 3 — The negation test: review 7, three ways

In [3]:
review_7 = df[df["id"] == 7]["review"].iloc[0]
review_7_cleaned = preprocessing.clean_text(review_7)
review_7_tokens = preprocessing.remove_stopwords(
    tokenizer.regex_word_tokenize(review_7_cleaned), nltk_stops
)
review_7_no_stops = " ".join(review_7_tokens)

print(f"RAW:            {review_7}")
print(f"CLEANED:        {review_7_cleaned}")
print(f"STOPWORDS OUT:  {review_7_no_stops}")
print()
print(f"VADER compound (RAW):           {sentiment.vader_score(review_7)['compound']}")
print(f"VADER compound (CLEANED):       {sentiment.vader_score(review_7_cleaned)['compound']}")
print(f"VADER compound (STOPWORDS OUT): {sentiment.vader_score(review_7_no_stops)['compound']}")


RAW:            Two hours and 15 mins of pure boredom. 2/10 would not recommend to anyone.
CLEANED:        two hours and mins of pure boredom would not recommend to anyone
STOPWORDS OUT:  two hours mins pure boredom would recommend anyone

VADER compound (RAW):           -0.5283
VADER compound (CLEANED):       -0.5283
VADER compound (STOPWORDS OUT): 0.0516


**This is the direct test of Practical 2's finding. Recall: Practical 2 showed "not" gets removed entirely by NLTK's default stopword list. Check whether that removal actually flips or dampens this specific review's score, or whether it turns out not to matter as much as expected.**

### Step 4 — VADER vs TextBlob across the whole dataset

In [4]:
results = []
for _, row in df.iterrows():
    scores = sentiment.compare_vader_textblob(row["review"])
    vader_label = sentiment.classify_compound(scores["vader"]["compound"])
    textblob_label = "positive" if scores["textblob"]["polarity"] > 0.05 else ("negative" if scores["textblob"]["polarity"] < -0.05 else "neutral")
    results.append({
        "id": row["id"],
        "review": row["review"],
        "vader_compound": scores["vader"]["compound"],
        "vader_label": vader_label,
        "textblob_polarity": scores["textblob"]["polarity"],
        "textblob_label": textblob_label,
        "agree": vader_label == textblob_label,
    })

results_df = pd.DataFrame(results)
results_df


,id,review,vader_compound,vader_label,textblob_polarity,textblob_label,agree
0,1,This movie was ABSOLUTELY fantastic!!! I've never seen anything like it before.,0.6588,positive,0.781250,positive,True
1,2,"Worst film of 2026. Don't waste your $12 on a ticket, I promise you won't like it.",-0.3773,negative,-0.600000,negative,True
2,3,"A solid 7/10 - great visuals, but the plot dragged on for way too long...",0.3716,positive,0.250000,positive,True
3,4,I can't believe how good the acting was!! Robert De Niro really outdid himself this time.,-0.4570,negative,0.300000,positive,False
4,5,"meh. it was fine i guess?? nothing special, wouldn't watch again tbh",-0.2773,negative,0.386905,positive,False
5,6,The director's 3rd project is by FAR his best work -- a must watch in theaters.,0.6369,positive,0.366667,positive,True
6,7,Two hours and 15 mins of pure boredom. 2/10 would not recommend to anyone.,-0.5283,negative,0.214286,positive,False
7,8,"WOW!!! Best film I've seen in years, hands down. 10/10 no notes.",0.8559,positive,0.346586,positive,True
8,9,"It's okay... not great, not terrible. Somewhere in the middle I'd say.",-0.5490,negative,0.150000,positive,False
9,10,Ticket prices are insane these days ($18.50!) but this one was worth every penny.,0.2003,positive,-0.350000,negative,False


In [5]:
agreement_rate = results_df["agree"].mean()
print(f"VADER/TextBlob label agreement: {agreement_rate:.0%} ({results_df['agree'].sum()}/{len(results_df)} reviews)")
print()
print("Reviews where they disagree:")
results_df[~results_df["agree"]][["id", "review", "vader_label", "textblob_label"]]


VADER/TextBlob label agreement: 60% (9/15 reviews)

Reviews where they disagree:


,id,review,vader_label,textblob_label
3,4,I can't believe how good the acting was!! Robert De Niro really outdid himself this time.,negative,positive
4,5,"meh. it was fine i guess?? nothing special, wouldn't watch again tbh",negative,positive
6,7,Two hours and 15 mins of pure boredom. 2/10 would not recommend to anyone.,negative,positive
8,9,"It's okay... not great, not terrible. Somewhere in the middle I'd say.",negative,positive
9,10,Ticket prices are insane these days ($18.50!) but this one was worth every penny.,positive,negative
13,14,5 stars from me! The soundtrack alone is worth the price of admission.,negative,positive


## Observations & Conclusion

Answer these based on what you actually saw when you ran the notebook:

- Was your Step 1 prediction correct? Did cleaning actually change VADER's compound score for reviews 1 and 8, and in which direction?
- For review 7 specifically: did removing "not" via stopword removal actually change the VADER compound score meaningfully, confirm the risk Practical 2 predicted, or turn out to matter less than expected? Give the three actual numbers.
- What was the VADER/TextBlob agreement rate across all 15 reviews? Look at any disagreements — do they happen on the reviews that seemed genuinely ambiguous to you as a human reader (like review 9, "not great, not terrible")?
- Pulling together everything from Practicals 1 through 9: what's your overall recommendation for how much preprocessing a sentiment analysis pipeline should actually do?

*(Answer)*
- The results show that VADER is sensitive to punctuation, capitalization, and emphasis, as positive reviews received lower sentiment scores after preprocessing removed features such as capital letters and repeated punctuation. Stopword removal had an even greater impact: removing negation words like "not" changed Review 7's sentiment from -0.5283 to 0.0516, demonstrating how essential negation is for accurate sentiment analysis. VADER and TextBlob agreed on 9 of the 15 reviews and disagreed on 6, but the disagreements revealed different weaknesses in each model rather than one consistently outperforming the other. VADER misclassified clearly positive reviews such as "I can't believe how good the acting was" and "5 stars from me!", while TextBlob struggled with reviews containing negation, mixed sentiment, or contrastive structures. For example, it correctly classified some positive reviews but incorrectly labeled "Ticket prices are insane... but this one was worth every penny" as negative, likely because it emphasized the negative word "insane" more than the positive clause following "but". Overall, the experiment demonstrates that lexicon-based sentiment analyzers can produce different interpretations of the same text, and that understanding context, negation, and discourse markers is crucial for reliable sentiment analysis.

---
## Viva Prep — Practice Questions

1. **What makes VADER specifically well-suited to informal/social-media text, compared to a general sentiment lexicon?**
   VADER's rules explicitly account for signals common in informal text - punctuation emphasis, capitalization, degree modifiers, negation, and some slang/emoticons - rather than treating all text as neutrally-formatted formal writing.

2. **Why would stopword removal specifically risk changing a VADER sentiment score, when VADER wasn't part of any earlier practical?**
   Because VADER's negation handling depends on words like "not" being present in the text it receives; if those words are removed upstream by stopword filtering before the text ever reaches VADER, the negation signal is gone regardless of how well VADER itself could have handled it.

3. **What's the practical difference between VADER's compound score and TextBlob's polarity score?**
   Both range roughly -1 to +1, but VADER's compound score is a rule-based aggregate that explicitly weights punctuation/caps/negation/intensifiers, while TextBlob's polarity comes from a simpler lexicon-average approach without those same rule-based adjustments.

4. **If two sentiment tools disagree on a review, does that necessarily mean one of them is "wrong"?**
   Not necessarily - some text is genuinely ambiguous or mixed in sentiment (e.g. "not great, not terrible"), and disagreement can reflect real ambiguity in the text rather than a straightforward error by either tool.

5. **Based on everything from Practicals 1-9, should a sentiment-analysis pipeline use the same cleaned text as a Bag-of-Words/TF-IDF pipeline?**
   Not necessarily - Practical 7 showed cleaning is generally fine for BoW/TF-IDF, but this practical is built to test whether the same aggressive cleaning measurably hurts a tool like VADER that depends on punctuation, capitalization, and intact negation words - reinforcing that preprocessing choices should be driven by what the specific downstream task actually needs, not applied uniformly.
